In [1]:
# Block 1: Imports & Loading Data
import pandas as pd
import os

print("Loading datasets...")

# Safe pathing configuration using os.path.join
data_path = "../data"
customers_file = os.path.join(data_path, "olist_customers_dataset.csv")
orders_file = os.path.join(data_path, "olist_orders_dataset.csv")

customers_df = pd.read_csv(customers_file)
orders_df = pd.read_csv(orders_file)

# 1. TYPE CASTING (Preventing Merge Crashes)
# Force all IDs to strictly be strings
customers_df['customer_id'] = customers_df['customer_id'].astype(str)
customers_df['customer_unique_id'] = customers_df['customer_unique_id'].astype(str)
orders_df['customer_id'] = orders_df['customer_id'].astype(str)

# 2. MISSING COLUMN DEFENSE
# If order_status is entirely missing from the new dataset, create a dummy one
if 'order_status' not in orders_df.columns:
    print("WARNING: 'order_status' column missing. Injecting default 'delivered' status.")
    orders_df['order_status'] = 'delivered'

# Verify the shape (Rows, Columns)
print(f"Customers table shape: {customers_df.shape}")
print(f"Orders table shape: {orders_df.shape}")

Loading datasets...
Customers table shape: (99441, 5)
Orders table shape: (99441, 8)


In [2]:
# Destroy any rows where the database failed to record the critical primary keys
customers_df = customers_df.dropna(subset=['customer_id', 'customer_unique_id'])
orders_df = orders_df.dropna(subset=['order_id', 'customer_id'])

#DEDUPLICATION (Actually dropping them)
initial_cust = len(customers_df)
initial_ord = len(orders_df)

customers_df = customers_df.drop_duplicates(subset=['customer_id'])
orders_df = orders_df.drop_duplicates(subset=['order_id'])

print(f"Dropped {initial_cust - len(customers_df)} duplicate customers.")
print(f"Dropped {initial_ord - len(orders_df)} duplicate orders.")

Dropped 0 duplicate customers.
Dropped 0 duplicate orders.


In [3]:
# Block 2: Merging and Cleaning Core Orders
print("Merging customers and orders...")

# 1. Merge the tables on the common key
merged_df = pd.merge(orders_df, customers_df, on='customer_id', how='inner')

# 2. Filter for successful purchases only (ignore 'cancelled' or 'unavailable')
merged_df = merged_df[merged_df['order_status'] == 'delivered']

# 3. Convert the timestamp from a string into a mathematical Datetime object
merged_df['order_purchase_timestamp'] = pd.to_datetime(merged_df['order_purchase_timestamp'])

# 4. Drop the heavy geographic string columns to save RAM. 
# We only need the Human, the Order, and the Date.
core_cols = ['customer_unique_id', 'order_id', 'order_purchase_timestamp']
rfm_base_df = merged_df[core_cols]

print(f"Base table ready. Cleaned Shape: {rfm_base_df.shape}")
display(rfm_base_df.head(3))

Merging customers and orders...
Base table ready. Cleaned Shape: (96478, 3)


,customer_unique_id,order_id,order_purchase_timestamp
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49


In [4]:
# Block 2.5: Post-Merge Null Annihilation (Safety Net)
print("--- POST-MERGE NULL CHECK & CLEANUP ---")

initial_len = len(rfm_base_df)

# We cannot 'fake' a purchase date. If it's missing, the row is dead.
rfm_base_df = rfm_base_df.dropna(subset=['order_purchase_timestamp'])

dropped_nulls = initial_len - len(rfm_base_df)
print(f"Dropped {dropped_nulls} rows due to missing timestamps.")

if dropped_nulls > 0:
    print(f"WARNING: The ingestion layer allowed {dropped_nulls} null timestamps through.")
else:
    print("Timestamp integrity verified. Ready for RFM math.")

--- POST-MERGE NULL CHECK & CLEANUP ---
Dropped 0 rows due to missing timestamps.
Timestamp integrity verified. Ready for RFM math.


In [5]:
# Block 3: Calculating Recency and Frequency
import datetime as dt

print("Calculating Recency and Frequency...")

# 1. Set the "Snapshot Date" (The anchor day to calculate 'days ago')
snapshot_date = rfm_base_df['order_purchase_timestamp'].max() + dt.timedelta(days=1)
print(snapshot_date)

# 2. Group by the human and calculate R (max date subtraction) and F (count)
rfm_df = rfm_base_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count'
}).reset_index()

# 3. Rename columns to the industry standard
rfm_df.rename(columns={
    'order_purchase_timestamp': 'Recency',
    'order_id': 'Frequency'
}, inplace=True)

print(f"RFM Table ready. Shape: {rfm_df.shape}")
display(rfm_df.head())

Calculating Recency and Frequency...
2018-08-30 15:00:37
RFM Table ready. Shape: (93358, 3)


,customer_unique_id,Recency,Frequency
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1
2,0000f46a3911fa3c0805444483337064,537,1
3,0000f6ccb0745a6a4b88665a16c9f078,321,1
4,0004aac84e0df4da2b147fca70cf8255,288,1


In [6]:
# Block 4: Exporting the RF Metrics
# Rename the dataframe to match reality
print("Exporting RF Metrics...")
rf_df = rfm_df.copy()

# SUGGESTION APPLIED: Ensure the target directory actually exists before saving
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "customer_rf_metrics.csv")

# Save locally using the os path
rf_df.to_csv(output_file, index=False)

print(f"RF metrics successfully exported to {output_file}!")
display(rf_df.head(2))
display(rf_df[rf_df['Frequency'] > 7])

Exporting RF Metrics...
RF metrics successfully exported to ../data\customer_rf_metrics.csv!


,customer_unique_id,Recency,Frequency
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1


,customer_unique_id,Recency,Frequency
22779,3e43e6105506432c953e165fb2acf44c,183,9
51431,8d50f5eadf50201ccdcedfb9e2ac8455,9,15
